In [1]:
import pandas as pd
import re
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import gc


In [2]:
df = pd.read_csv("text_classifier_dataset.csv", on_bad_lines='skip', engine='python')

# 🔥 Reduce data size (POC training)
df = df.sample(2000, random_state=42).reset_index(drop=True)

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\w+|#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)


In [4]:
LABELS = [
    "billing",
    "technical",
    "account",
    "product",
    "delivery",
    "feedback",
    "spam"
]


In [5]:
# Initialize label columns
for label in LABELS:
    df[label] = 0


In [6]:
def assign_labels(text):
    t = text.lower()
    labels = []

    if any(w in t for w in ["payment", "refund", "charged"]):
        labels.append("billing")
    if any(w in t for w in ["error", "crash", "bug", "not working"]):
        labels.append("technical")
    if any(w in t for w in ["login", "account", "password"]):
        labels.append("account")
    if any(w in t for w in ["broken", "quality", "defective"]):
        labels.append("product")
    if any(w in t for w in ["delay", "late", "delivery"]):
        labels.append("delivery")
    if any(w in t for w in ["thanks", "great", "love", "good"]):
        labels.append("feedback")
    if any(w in t for w in ["http", "buy now", "free", "offer"]):
        labels.append("spam")

    if not labels:
        labels.append("feedback")

    return labels

df["labels"] = df["clean_text"].apply(assign_labels)

for label in LABELS:
    df[label] = df["labels"].apply(lambda x: int(label in x))


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df["clean_text"],
    df[LABELS].values,
    test_size=0.2,
    random_state=42
)


In [8]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

train_enc = tokenize(X_train)   # ✅ THIS LINE IS REQUIRED


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=64,   # 🔥 MUST
        return_tensors="pt"
    )

train_enc = tokenize(X_train)


In [10]:
class MultiLabelDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)


In [11]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=7,
    problem_type="multi_label_classification"
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
device = torch.device("cpu")   # 🔥 force CPU
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

train_loader = DataLoader(
    MultiLabelDataset(train_enc, y_train),
    batch_size=1,   # 🔥 MUST
    shuffle=True
)

model.train()
for epoch in range(2):
    loop = tqdm(train_loader)
    for batch in loop:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())
    gc.collect()


100%|██████████| 1600/1600 [14:58<00:00,  1.78it/s, loss=2.88e-5] 


In [14]:
import gc
gc.collect()

model.save_pretrained(
    "src/models/bert_multilabel",
    safe_serialization=False
)

tokenizer.save_pretrained("src/models/bert_multilabel")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: The requested operation cannot be performed on a file with a user-mapped section open. (os error 1224)